# 03b — Optimized Baseline (stock YOLOv8n, tuned to chase the paper's 0.774)

Same **architecture** as notebook 03 (stock YOLOv8n = the paper's baseline) — we only stack
**accuracy-oriented training/inference techniques** to close the gap to the paper's baseline
**mAP@0.5 = 0.774**. The paper publishes no hyperparameters, so these are the standard,
architecture-preserving levers for a small, low-contrast defect dataset:

| Technique | Why |
|---|---|
| **imgsz 640 → 800** | more resolution for small/low-contrast defects (crazing, rolled-in_scale) |
| **SGD + cosine LR, 200 epochs, patience 60** | train fully to convergence |
| **close_mosaic=20, mixup=0.1** | clean final epochs sharpen localization; mild regularization |
| **TTA (augment=True) at eval** | flip/scale ensembling at inference — free accuracy |
| **NMS IoU=0.6** | the paper's dynamic-NMS tweak for dense small targets |

> This is a *separate experiment* aimed at matching the paper's number — it uses a heavier recipe
> than the fair 3-way comparison (notebooks 03/05/06). Results → `results/baseline_opt/`.
> ⚠️ At imgsz=800 this run is slow (~4–5 h for 200 epochs). Set `IMGSZ=640`, `BATCH=16` in the
> train cell for a ~3× faster run if you don't need the extra resolution.

## 1. Check environment
> **Kernel:** `.venv (Python 3.10.8)` — the venv with Ultralytics.

In [1]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation


## 2. Locate dataset config

In [2]:
from pathlib import Path
from ultralytics import YOLO
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_CFG = ROOT / 'data' / 'neu-det-yolo' / 'data.yaml'
assert DATA_CFG.exists(), 'Run 01_data_preparation.ipynb first!'
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
print('Data:', DATA_CFG)

Data: c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml


## 3. Load pretrained YOLOv8n (same architecture as the paper baseline)

In [3]:
model = YOLO('yolov8n.pt')   # COCO-pretrained stock YOLOv8n (3.01M params / 8.1 GFLOPs)
print('Loaded YOLOv8n')

Loaded YOLOv8n


## 4. Train (optimized recipe)
Stock YOLOv8n + the accuracy stack above. `workers=0` is Windows-safe (no `close_mosaic` deadlock).
> **VRAM:** imgsz=800 + batch=8 fits ~16 GB. If you OOM, drop `BATCH` to 4. For a fast run use
> `IMGSZ=640, BATCH=16`.

In [4]:
IMGSZ = 800              # <- main lever; set 640 for ~3x faster training
BATCH = 8 if DEVICE == 0 else 2    # 800px+16GB; drop to 4 if OOM (or 16 if IMGSZ=640)

results = model.train(
    data=str(DATA_CFG),
    epochs=200,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    cache=True,
    workers=0,            # Windows-safe
    project=str(ROOT / 'results'),
    name='baseline_opt',
    exist_ok=True,
    optimizer='SGD',      # SGD+momentum (lr0=0.01) - standard detection optimizer
    cos_lr=True,          # cosine LR decay
    patience=60,          # generous: don't cut the long run before it converges
    close_mosaic=20,      # last 20 epochs train on clean (non-mosaic) images
    mixup=0.1,            # mild regularization for the small dataset
    seed=42,
    plots=True,
)
print('Training done. Best weights:', ROOT / 'results/baseline_opt/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n

## 5. Evaluate — ablation: plain vs optimized (TTA + NMS 0.6) on the TEST set
Both reload `best.pt` so this runs without re-training. The optimized eval adds **TTA**
(`augment=True`) and the paper's **NMS IoU=0.6**. Target: paper baseline **0.774**.

In [5]:
best = ROOT / 'results' / 'baseline_opt' / 'weights' / 'best.pt'

m = YOLO(str(best))
plain = m.val(data=str(DATA_CFG), split='test', verbose=False)
opt   = YOLO(str(best)).val(data=str(DATA_CFG), split='test', augment=True, iou=0.6, verbose=False)

print(f"{'eval':<22}{'mAP@0.5':>10}{'mAP@.5:.95':>12}{'P':>8}{'R':>8}")
print(f"{'plain':<22}{plain.box.map50:>10.4f}{plain.box.map:>12.4f}{plain.box.mp:>8.3f}{plain.box.mr:>8.3f}")
print(f"{'optimized (TTA+NMS.6)':<22}{opt.box.map50:>10.4f}{opt.box.map:>12.4f}{opt.box.mp:>8.3f}{opt.box.mr:>8.3f}")
print(f"\npaper baseline target: 0.774   |   best here: {max(plain.box.map50, opt.box.map50):.4f}")

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 49.324.2 MB/s, size: 13.6 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 5.8it/s 2.1s0.1s
                   all        180        413      0.685      0.677      0.716       0.38
Speed: 2.8ms preprocess, 4.9ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\student\Desktop\SteelDefectDetection\notebooks\runs\detect\val-15
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GF

## 6. Per-class table (best eval) & save summary
Writes `results/baseline_opt/metrics_summary.txt`.

In [6]:
import pandas as pd
best_res = opt if float(opt.box.map50) >= float(plain.box.map50) else plain
tag = 'optimized (TTA+NMS0.6)' if best_res is opt else 'plain'

rows = [{'class': n, 'mAP@0.5': round(float(best_res.box.ap50[i]), 4),
         'mAP@0.5:0.95': round(float(best_res.box.ap[i]), 4)} for i, n in enumerate(CLASSES)]
df = pd.DataFrame(rows)
df.loc[len(df)] = ['ALL (mean)', round(float(best_res.box.map50), 4), round(float(best_res.box.map), 4)]

run_dir = ROOT / 'results' / 'baseline_opt'; run_dir.mkdir(parents=True, exist_ok=True)
summary = run_dir / 'metrics_summary.txt'
with open(summary, 'w') as f:
    f.write('Optimized Baseline YOLOv8n (imgsz=800, SGD, 200ep, TTA+NMS) - TEST set\n')
    f.write('=' * 70 + '\n')
    f.write(f'best eval    : {tag}\n')
    f.write(f'mAP@0.5      : {best_res.box.map50:.4f}  (paper baseline 0.774)\n')
    f.write(f'mAP@0.5:0.95 : {best_res.box.map:.4f}\n')
    f.write(f'precision    : {best_res.box.mp:.4f}\n')
    f.write(f'recall       : {best_res.box.mr:.4f}\n\nPer-class mAP@0.5:\n')
    for i, n in enumerate(CLASSES):
        f.write(f'  {n:<18}{float(best_res.box.ap50[i]):.4f}\n')
print('saved', summary)
df

saved c:\Users\student\Desktop\SteelDefectDetection\results\baseline_opt\metrics_summary.txt


,class,mAP@0.5,mAP@0.5:0.95
0,crazing,0.5751,0.2232
1,inclusion,0.8276,0.4485
2,patches,0.9222,0.5717
3,pitted_surface,0.8391,0.4489
4,rolled-in_scale,0.5665,0.2248
5,scratches,0.8476,0.3732
6,ALL (mean),0.7630,0.3817


✅ **Optimized baseline done.** Weights → `results/baseline_opt/weights/best.pt`, metrics →
`results/baseline_opt/metrics_summary.txt`. Compare the best test mAP@0.5 to the paper's **0.774**
and to the plain baseline (notebook 03). If still short, the next levers are `IMGSZ=960` or
`epochs=300`.